# Preprocessing

Load the six raw CSVs, unify the schema, and save two processed files:

- `kb_combined.csv` — the knowledge base (MedQuAD + MedlinePlus + PubMedQA + MedRedQA)
- `eval_combined.csv` — the evaluation set (PubMedQA labeled + HealthSearchQA)

Unified schema:

| Field | Description |
|-------|-------------|
| `doc_id` | unique within the file |
| `source` | medquad / medlineplus / pubmedqa / medredqa |
| `source_id` | original id from the raw source |
| `category` | qtype when available, else empty |
| `question` | the question (synthesized for MedlinePlus) |
| `context` | background text (abstract, Reddit body, etc.) |
| `answer` | the answer / summary |

In [ ]:
import re
import ast
import pandas as pd
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(sorted(f.name for f in RAW_DIR.glob("*.csv")))

['healthsearchqa_raw.csv', 'medlineplus_raw.csv', 'medquad_raw.csv', 'medredqa_raw.csv', 'pubmedqa_eval_raw.csv', 'pubmedqa_kb_raw.csv']


In [ ]:
def clean_text(t):
    if t is None:
        return ""
    t = str(t)
    t = re.sub(r'<[^>]+>', ' ', t)      # strip html tags
    t = re.sub(r'\s+', ' ', t)          # collapse whitespace
    t = re.sub(r'\s+\?', '?', t)        # space before question mark
    t = re.sub(r'\?+', '?', t)          # collapse multiple ?
    t = t.replace(" .", ".").replace(" ,", ",")
    return t.strip()


def parse_context(x):
    """Extract the 'contexts' list from a PubMedQA context column.
    Handles: dict, Python-literal string, and stringified numpy array.
    """
    if isinstance(x, dict):
        return " ".join(str(c) for c in x.get("contexts", []))

    s = str(x)

    # Case 1: plain literal string like "{'contexts': [...]}"
    try:
        d = ast.literal_eval(s)
        if isinstance(d, dict) and "contexts" in d:
            return " ".join(str(c) for c in d["contexts"])
    except (ValueError, SyntaxError):
        pass

    # Case 2: stringified numpy array like "{'contexts': array([...])}"
    match = re.search(
        r"'contexts':\s*array\(\[(.*?)\](?:,\s*dtype=[^)]+)?\)",
        s,
        re.DOTALL,
    )
    if match:
        inner = match.group(1).strip()
        if inner:
            parts = re.findall(r"'((?:[^'\\]|\\.)*)'", inner)
            if parts:
                return " ".join(parts)

    return s

## 1. MedQuAD

Already has a `qtype` we can use as the category.

In [ ]:
raw = pd.read_csv(RAW_DIR / "medquad_raw.csv")

mq = pd.DataFrame({
    "source": "medquad",
    "source_id": raw.index.astype(str),
    "category": raw["qtype"],
    "question": raw["Question"],
    "context": "",
    "answer": raw["Answer"],
})

print(len(mq))

16407


## 2. MedlinePlus

No question field — we synthesize one from the title.

In [ ]:
raw = pd.read_csv(RAW_DIR / "medlineplus_raw.csv")

ml = pd.DataFrame({
    "source": "medlineplus",
    "source_id": raw["url"],
    "category": "",
    "question": "What is " + raw["title"].astype(str) + "?",
    "context": "",
    "answer": raw["summary"],
})

print(len(ml))

1017


## 3. PubMedQA (KB)

The `context` field is the PubMed abstract. Apply `parse_context` to
unwrap the stringified numpy array.

In [ ]:
raw = pd.read_csv(RAW_DIR / "pubmedqa_kb_raw.csv")

pq = pd.DataFrame({
    "source": "pubmedqa",
    "source_id": raw["pubid"].astype(str),
    "category": "",
    "question": raw["question"],
    "context": raw["context"],
    "answer": raw["long_answer"],
})

pq["context"] = pq["context"].apply(parse_context)

print(len(pq))
print(pq["context"].iloc[0][:150])

<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1:

211269
Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently dis


## 4. MedRedQA

Question = title, context = user's description, answer = physician response.
Drop very short answers — they are usually casual replies that don't carry medical knowledge.

In [ ]:
raw = pd.read_csv(RAW_DIR / "medredqa_raw.csv")

mr = pd.DataFrame({
    "source": "medredqa",
    "source_id": raw["id"].astype(str),
    "category": "",
    "question": raw["title"],
    "context": raw["body"],
    "answer": raw["response"],
})

print("before filter:", len(mr))

mr = mr[mr["answer"].astype(str).str.len() >= 100].reset_index(drop=True)
print("after filter:", len(mr))

before filter: 40602
after filter: 29801


## 5. Combine and clean

In [ ]:
kb = pd.concat([mq, ml, pq, mr], ignore_index=True)

for col in ["question", "context", "answer", "category"]:
    kb[col] = kb[col].apply(clean_text)

# keep rows that have at least an answer or a context
kb = kb[(kb["answer"] != "") | (kb["context"] != "")].reset_index(drop=True)
kb.insert(0, "doc_id", kb.index.astype(str))

print("total:", len(kb))
print(kb["source"].value_counts())

total: 258494
source
pubmedqa       211269
medredqa        29801
medquad         16407
medlineplus      1017
Name: count, dtype: int64


In [ ]:
kb.head(3)

,doc_id,source,source_id,category,question,context,answer
0,0,medquad,0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,,LCMV infections can occur after exposure to fr...
1,1,medquad,1,symptoms,What are the symptoms of Lymphocytic Choriomen...,,LCMV is most commonly recognized as causing ne...
2,2,medquad,2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,,Individuals of all ages who come into contact ...


In [ ]:
kb["has_answer"] = kb["answer"] != ""
kb["has_context"] = kb["context"] != ""
kb.groupby("source")[["has_answer", "has_context"]].mean().round(3)

,has_answer,has_context
source,,
medlineplus,1.0,0.0
medquad,1.0,0.0
medredqa,1.0,1.0
pubmedqa,1.0,1.0


## 6. Eval set

PubMedQA labeled + HealthSearchQA.

In [ ]:
raw = pd.read_csv(RAW_DIR / "pubmedqa_eval_raw.csv")

pq_eval = pd.DataFrame({
    "source": "pubmedqa_eval",
    "source_id": raw["pubid"].astype(str),
    "question": raw["question"],
    "context": raw["context"],
    "answer": raw["long_answer"],
})

pq_eval["context"] = pq_eval["context"].apply(parse_context)
print(pq_eval["context"].iloc[0][:150])

Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in 


In [ ]:
raw = pd.read_csv(RAW_DIR / "healthsearchqa_raw.csv")

hs = pd.DataFrame({
    "source": "healthsearchqa",
    "source_id": raw["id"].astype(str),
    "question": raw["question"],
    "context": "",
    "answer": "",
})

eval_set = pd.concat([pq_eval, hs], ignore_index=True)

for col in ["question", "context", "answer"]:
    eval_set[col] = eval_set[col].apply(clean_text)

eval_set = eval_set[eval_set["question"] != ""].reset_index(drop=True)
eval_set.insert(0, "eval_id", eval_set.index.astype(str))

print("total:", len(eval_set))
print(eval_set["source"].value_counts())

total: 5436
source
healthsearchqa    4436
pubmedqa_eval     1000
Name: count, dtype: int64


In [ ]:
eval_set.head(3)

,eval_id,source,source_id,question,context,answer
0,0,pubmedqa_eval,21645374,Do mitochondria play a role in remodelling lac...,Programmed cell death (PCD) is the regulated d...,Results depicted mitochondrial dynamics in viv...
1,1,pubmedqa_eval,16418930,Landolt C and snellen e acuity: differences in...,Assessment of visual acuity depends on the opt...,"Using the charts described, there was only a s..."
2,2,pubmedqa_eval,9488747,"Syncope during bathing in infants, a pediatric...",Apparent life-threatening events in infants ar...,"""Aquagenic maladies"" could be a pediatric form..."


## 7. Save

In [ ]:
kb.to_csv(PROCESSED_DIR / "kb_combined.csv", index=False)
eval_set.to_csv(PROCESSED_DIR / "eval_combined.csv", index=False)

print("kb_combined.csv:", len(kb))
print("eval_combined.csv:", len(eval_set))

kb_combined.csv: 258494
eval_combined.csv: 5436


Next: `03_category_labelling.ipynb`